# Olah Data Validasi Produk

Notebook ini mengolah data validasi ahli/praktisi pada folder `data/validasi`.

Output utama:
- ringkasan skor Likert dan persentase kelayakan;
- Aiken's V per butir, per aspek, dan keseluruhan;
- ringkasan catatan kualitatif, kritik, dan saran;
- file siap pakai di `outputs/validasi/`.

Catatan: validasi ahli menggunakan skala 1-4. Rumus Aiken's V yang dipakai adalah `V = sum(r - lo) / (n * (c - 1))`, dengan `lo = 1`, `c = 4`, dan `n` jumlah penilai yang skornya terisi. Butir/aspek dinyatakan valid jika `V >= 0,78`.

In [ ]:
from pathlib import Path
import csv
import math
from statistics import mean
from textwrap import shorten

def find_project_root():
    cwd = Path.cwd().resolve()
    for path in [cwd, *cwd.parents]:
        if (path / 'data' / 'validasi').exists():
            return path
    raise FileNotFoundError('Folder data/validasi tidak ditemukan dari current working directory.')

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / 'data' / 'validasi'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'validasi'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCALE_MIN = 1
SCALE_MAX = 4

print('Project root:', PROJECT_ROOT)
print('Data dir    :', DATA_DIR)
print('Output dir  :', OUTPUT_DIR)

## Fungsi Bantu

In [ ]:
def read_csv(path):
    with path.open(newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))

def write_csv(path, rows, fieldnames):
    with path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

def to_float(value):
    if value is None:
        return None
    text = str(value).strip().replace(',', '.')
    if not text:
        return None
    try:
        return float(text)
    except ValueError:
        return None

def score_columns(rows):
    if not rows:
        return []
    columns = rows[0].keys()
    prefixes = ('validator_', 'praktisi_', 'ahli_')
    return [col for col in columns if col.startswith(prefixes)]

def valid_scores(row, cols):
    scores = []
    for col in cols:
        score = to_float(row.get(col))
        if score is not None:
            scores.append(score)
    return scores

def pct_category(value):
    if value is None:
        return 'Belum Ada Data'
    if value >= 81:
        return 'Sangat Valid'
    if value >= 61:
        return 'Valid'
    if value >= 41:
        return 'Cukup Valid'
    if value >= 21:
        return 'Kurang Valid'
    return 'Tidak Valid'

def aiken_category(value):
    if value is None:
        return 'Belum Ada Data'
    if value >= 0.78:
        return 'Valid'
    return 'Perlu Revisi'

def fmt_num(value, digits=2):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ''
    return f'{value:.{digits}f}'

def markdown_table(rows, columns):
    if not rows:
        return '_Tidak ada data._'
    lines = []
    lines.append('| ' + ' | '.join(columns) + ' |')
    lines.append('| ' + ' | '.join(['---'] * len(columns)) + ' |')
    for row in rows:
        values = [str(row.get(col, '')).replace('\n', '<br>') for col in columns]
        lines.append('| ' + ' | '.join(values) + ' |')
    return '\n'.join(lines)

def title_from_filename(path):
    name = path.stem.replace('validasi_', '').replace('_', ' ')
    return name.title()

def aspek_utama(indikator):
    text = (indikator or '').strip()
    if ' - ' in text:
        return text.split(' - ', 1)[0].strip()
    return text

## Muat Data Validasi Kuantitatif

In [ ]:
validation_files = sorted(
    path for path in DATA_DIR.glob('validasi_*.csv')
    if not path.stem.endswith('_catatan')
)

datasets = []
for path in validation_files:
    rows = read_csv(path)
    cols = score_columns(rows)
    datasets.append({'path': path, 'nama': title_from_filename(path), 'rows': rows, 'score_cols': cols})

for ds in datasets:
    filled = sum(len(valid_scores(row, ds['score_cols'])) for row in ds['rows'])
    possible = len(ds['rows']) * len(ds['score_cols'])
    print(f"{ds['nama']}: {len(ds['rows'])} butir, kolom skor={ds['score_cols']}, skor terisi={filled}/{possible}")

if not datasets:
    print('Tidak ada file validasi_*.csv yang bisa diolah.')

## Hitung Persentase dan Aiken's V

In [ ]:
detail_rows = []
aspect_rows = []
summary_rows = []

for ds in datasets:
    nama = ds['nama']
    rows = ds['rows']
    cols = ds['score_cols']
    aspek_groups = {}
    all_scores = []

    for row in rows:
        scores = valid_scores(row, cols)
        all_scores.extend(scores)
        n = len(scores)
        skor_total = sum(scores) if scores else None
        skor_maks = n * SCALE_MAX if n else None
        persentase = (skor_total / skor_maks * 100) if skor_maks else None
        aiken_v = (sum(score - SCALE_MIN for score in scores) / (n * (SCALE_MAX - SCALE_MIN))) if n else None
        indikator = row.get('indikator', '').strip()
        aspek = aspek_utama(indikator)
        aspek_groups.setdefault(aspek, []).append({'scores': scores, 'persentase': persentase, 'aiken_v': aiken_v})

        detail_rows.append({
            'jenis_validasi': nama,
            'no': row.get('no', ''),
            'aspek': aspek,
            'indikator': indikator,
            'pernyataan': row.get('pernyataan', ''),
            'n_penilai': n,
            'skor_total': fmt_num(skor_total, 0),
            'skor_maks': fmt_num(skor_maks, 0),
            'persentase': fmt_num(persentase),
            'kategori_persentase': pct_category(persentase),
            'aiken_v': fmt_num(aiken_v, 3),
            'kategori_aiken': aiken_category(aiken_v),
        })

    for aspek, items in aspek_groups.items():
        scores = [score for item in items for score in item['scores']]
        n_scores = len(scores)
        skor_total = sum(scores) if scores else None
        skor_maks = n_scores * SCALE_MAX if n_scores else None
        persentase = (skor_total / skor_maks * 100) if skor_maks else None
        aiken_values = [item['aiken_v'] for item in items if item['aiken_v'] is not None]
        aiken_mean = mean(aiken_values) if aiken_values else None
        aspect_rows.append({
            'jenis_validasi': nama,
            'aspek': aspek,
            'jumlah_butir': len(items),
            'n_skor_terisi': n_scores,
            'skor_total': fmt_num(skor_total, 0),
            'skor_maks': fmt_num(skor_maks, 0),
            'persentase': fmt_num(persentase),
            'kategori_persentase': pct_category(persentase),
            'aiken_v_rerata': fmt_num(aiken_mean, 3),
            'kategori_aiken': aiken_category(aiken_mean),
        })

    n_scores = len(all_scores)
    skor_total = sum(all_scores) if all_scores else None
    skor_maks = n_scores * SCALE_MAX if n_scores else None
    persentase = (skor_total / skor_maks * 100) if skor_maks else None
    aiken_v = (sum(score - SCALE_MIN for score in all_scores) / (n_scores * (SCALE_MAX - SCALE_MIN))) if n_scores else None
    missing_cols = []
    for col in cols:
        if sum(1 for row in rows if to_float(row.get(col)) is not None) == 0:
            missing_cols.append(col)
    summary_rows.append({
        'jenis_validasi': nama,
        'jumlah_butir': len(rows),
        'kolom_penilai': ', '.join(cols),
        'n_skor_terisi': n_scores,
        'skor_total': fmt_num(skor_total, 0),
        'skor_maks': fmt_num(skor_maks, 0),
        'persentase': fmt_num(persentase),
        'kategori_persentase': pct_category(persentase),
        'aiken_v': fmt_num(aiken_v, 3),
        'kategori_aiken': aiken_category(aiken_v),
        'catatan_data': 'Kolom kosong: ' + ', '.join(missing_cols) if missing_cols else '',
    })

print(markdown_table(summary_rows, ['jenis_validasi', 'jumlah_butir', 'n_skor_terisi', 'skor_total', 'skor_maks', 'persentase', 'kategori_persentase', 'aiken_v', 'kategori_aiken']))

## Ringkasan Per Aspek

In [ ]:
print(markdown_table(aspect_rows, ['jenis_validasi', 'aspek', 'jumlah_butir', 'skor_total', 'skor_maks', 'persentase', 'kategori_persentase', 'aiken_v_rerata', 'kategori_aiken']))

## Olah Catatan Kualitatif

In [ ]:
catatan_rows = []
for path in sorted(DATA_DIR.glob('validasi_*_catatan.csv')):
    jenis = title_from_filename(Path(path.stem.replace('_catatan', '.csv')))
    for row in read_csv(path):
        catatan = (row.get('catatan') or '').strip()
        if catatan:
            catatan_rows.append({
                'jenis_validasi': jenis,
                'aspek': row.get('aspek', '').strip(),
                'catatan': catatan,
            })

revisi_rows = []
revisi_path = DATA_DIR / 'revisi_produk.csv'
if revisi_path.exists():
    for row in read_csv(revisi_path):
        revisi_rows.append(row)

print('Catatan kualitatif terisi:', len(catatan_rows))
print('Masukan/revisi produk   :', len(revisi_rows))
print(markdown_table(catatan_rows, ['jenis_validasi', 'aspek', 'catatan']))

## Buat Narasi Ringkas

In [ ]:
def row_float(row, key):
    return to_float(row.get(key))

def narrative_for_summary(row):
    jenis = row['jenis_validasi']
    pct = row.get('persentase', '')
    cat = row.get('kategori_persentase', '')
    v = row.get('aiken_v', '')
    vcat = row.get('kategori_aiken', '')
    aspects = [r for r in aspect_rows if r['jenis_validasi'] == jenis and row_float(r, 'persentase') is not None]
    if aspects:
        highest = max(aspects, key=lambda r: row_float(r, 'persentase'))
        lowest = min(aspects, key=lambda r: row_float(r, 'persentase'))
        aspect_text = (
            f"Aspek tertinggi adalah {highest['aspek']} ({highest['persentase']}%), "
            f"sedangkan aspek terendah adalah {lowest['aspek']} ({lowest['persentase']}%)."
        )
    else:
        aspect_text = 'Belum ada aspek yang dapat diringkas.'
    return (
        f"Validasi {jenis.lower()} memperoleh skor {row['skor_total']} dari skor maksimum {row['skor_maks']} "
        f"atau {pct}% dengan kategori {cat}. Nilai Aiken's V keseluruhan adalah {v} "
        f"dengan interpretasi deskriptif {vcat}. {aspect_text}"
    )

narasi_rows = []
for row in summary_rows:
    narasi_rows.append({'jenis_validasi': row['jenis_validasi'], 'narasi': narrative_for_summary(row)})

for row in narasi_rows:
    print(f"\n## {row['jenis_validasi']}\n{row['narasi']}")

## Simpan Output

In [ ]:
write_csv(OUTPUT_DIR / 'validasi_detail_butir.csv', detail_rows, ['jenis_validasi', 'no', 'aspek', 'indikator', 'pernyataan', 'n_penilai', 'skor_total', 'skor_maks', 'persentase', 'kategori_persentase', 'aiken_v', 'kategori_aiken'])
write_csv(OUTPUT_DIR / 'validasi_ringkasan_aspek.csv', aspect_rows, ['jenis_validasi', 'aspek', 'jumlah_butir', 'n_skor_terisi', 'skor_total', 'skor_maks', 'persentase', 'kategori_persentase', 'aiken_v_rerata', 'kategori_aiken'])
write_csv(OUTPUT_DIR / 'validasi_rekap.csv', summary_rows, ['jenis_validasi', 'jumlah_butir', 'kolom_penilai', 'n_skor_terisi', 'skor_total', 'skor_maks', 'persentase', 'kategori_persentase', 'aiken_v', 'kategori_aiken', 'catatan_data'])
write_csv(OUTPUT_DIR / 'validasi_narasi.csv', narasi_rows, ['jenis_validasi', 'narasi'])

md = []
md.append('# Ringkasan Hasil Validasi Produk')
md.append('')
md.append('## Rekapitulasi Kuantitatif')
md.append(markdown_table(summary_rows, ['jenis_validasi', 'jumlah_butir', 'n_skor_terisi', 'skor_total', 'skor_maks', 'persentase', 'kategori_persentase', 'aiken_v', 'kategori_aiken']))
md.append('')
md.append('## Analisis Per Aspek')
md.append(markdown_table(aspect_rows, ['jenis_validasi', 'aspek', 'jumlah_butir', 'skor_total', 'skor_maks', 'persentase', 'kategori_persentase', 'aiken_v_rerata', 'kategori_aiken']))
md.append('')
md.append('## Narasi Ringkas')
for row in narasi_rows:
    md.append(f"### {row['jenis_validasi']}")
    md.append(row['narasi'])
    md.append('')
md.append('## Catatan Kualitatif Validator')
md.append(markdown_table(catatan_rows, ['jenis_validasi', 'aspek', 'catatan']))
md.append('')
if revisi_rows:
    md.append('## Revisi Produk Berdasarkan Masukan')
    md.append(markdown_table(revisi_rows, ['no', 'sumber', 'masukan_validator', 'revisi_yang_dilakukan', 'keterangan_gambar']))
    md.append('')
md.append('## Catatan Metodologis')
md.append("Persentase dihitung dari skor terisi dibanding skor maksimum skala 1-4. Aiken's V dihitung dengan lo=1 dan c=4. Jika hanya satu penilai terisi, nilai V tetap dapat dihitung secara mekanis, tetapi interpretasinya perlu dibaca hati-hati karena validitas isi idealnya melibatkan beberapa ahli.")

(OUTPUT_DIR / 'ringkasan_validasi.md').write_text('\n'.join(md), encoding='utf-8')

for path in sorted(OUTPUT_DIR.glob('validasi_*')) + [OUTPUT_DIR / 'ringkasan_validasi.md']:
    print(path.relative_to(PROJECT_ROOT))